EDI Transformation

In [0]:
spark

In [0]:
file_path = "/Volumes/workspace/default/edi_project_volume/850_Sample copy_github.txt"

df = spark.read.text(file_path)

df.show(truncate=False)

+---------------------------------------------------------------------------------------------------------------+
|value                                                                                                          |
+---------------------------------------------------------------------------------------------------------------+
|ISA*00*           *00*           *ZZ*SENDERID        *ZZ*RECEIVERID      *260806*1000*U*00401*000000001*0*T*>~ |
|GS*PO*SENDERID*RECEIVERID*20260806*1000*1*X*004010~                                                            |
|ST*850*0001~                                                                                                   |
|BEG*00*NE*PO20260806**20260806~                                                                                |
|REF*DP*001~                                                                                                    |
|N1*BY*NORTHSTAR RETAIL~                                                                

In [0]:
df.printSchema()

root
 |-- value: string (nullable = true)



In [0]:
df.show(5, truncate = False)

+---------------------------------------------------------------------------------------------------------------+
|value                                                                                                          |
+---------------------------------------------------------------------------------------------------------------+
|ISA*00*           *00*           *ZZ*SENDERID        *ZZ*RECEIVERID      *260806*1000*U*00401*000000001*0*T*>~ |
|GS*PO*SENDERID*RECEIVERID*20260806*1000*1*X*004010~                                                            |
|ST*850*0001~                                                                                                   |
|BEG*00*NE*PO20260806**20260806~                                                                                |
|REF*DP*001~                                                                                                    |
+---------------------------------------------------------------------------------------

In [0]:
edi_df = df.withColumnRenamed("value", "edi_segment")
edi_df.show(truncate = False)

+---------------------------------------------------------------------------------------------------------------+
|edi_segment                                                                                                    |
+---------------------------------------------------------------------------------------------------------------+
|ISA*00*           *00*           *ZZ*SENDERID        *ZZ*RECEIVERID      *260806*1000*U*00401*000000001*0*T*>~ |
|GS*PO*SENDERID*RECEIVERID*20260806*1000*1*X*004010~                                                            |
|ST*850*0001~                                                                                                   |
|BEG*00*NE*PO20260806**20260806~                                                                                |
|REF*DP*001~                                                                                                    |
|N1*BY*NORTHSTAR RETAIL~                                                                

In [0]:
from pyspark.sql.functions import split

parsed_df = edi_df.withColumn(
    "segment_type",
    split("edi_segment", "\\*")[0]
)

parsed_df.show(truncate=False)

+---------------------------------------------------------------------------------------------------------------+------------+
|edi_segment                                                                                                    |segment_type|
+---------------------------------------------------------------------------------------------------------------+------------+
|ISA*00*           *00*           *ZZ*SENDERID        *ZZ*RECEIVERID      *260806*1000*U*00401*000000001*0*T*>~ |ISA         |
|GS*PO*SENDERID*RECEIVERID*20260806*1000*1*X*004010~                                                            |GS          |
|ST*850*0001~                                                                                                   |ST          |
|BEG*00*NE*PO20260806**20260806~                                                                                |BEG         |
|REF*DP*001~                                                                                                   

In [0]:
parsed_df2 = edi_df.withColumn("segment_one",split("edi_segment","\\*")[1])
parsed_df2.show(truncate=False)

+---------------------------------------------------------------------------------------------------------------+-----------+
|edi_segment                                                                                                    |segment_one|
+---------------------------------------------------------------------------------------------------------------+-----------+
|ISA*00*           *00*           *ZZ*SENDERID        *ZZ*RECEIVERID      *260806*1000*U*00401*000000001*0*T*>~ |00         |
|GS*PO*SENDERID*RECEIVERID*20260806*1000*1*X*004010~                                                            |PO         |
|ST*850*0001~                                                                                                   |850        |
|BEG*00*NE*PO20260806**20260806~                                                                                |00         |
|REF*DP*001~                                                                                                    |DP   

In [0]:
elements_df = parsed_df.withColumn("elements",split("edi_segment","\\*"))
elements_df.show(truncate=False)

+---------------------------------------------------------------------------------------------------------------+------------+---------------------------------------------------------------------------------------------------------------------------------+
|edi_segment                                                                                                    |segment_type|elements                                                                                                                         |
+---------------------------------------------------------------------------------------------------------------+------------+---------------------------------------------------------------------------------------------------------------------------------+
|ISA*00*           *00*           *ZZ*SENDERID        *ZZ*RECEIVERID      *260806*1000*U*00401*000000001*0*T*>~ |ISA         |[ISA, 00,            , 00,            , ZZ, SENDERID        , ZZ, RECEIVERID      , 260806, 1000, U, 00

In [0]:
po1_df = elements_df.filter(
    elements_df["segment_type"] == "PO1"
)

po1_df.show(truncate=False)

+---------------------------------+------------+------------------------------------------+
|edi_segment                      |segment_type|elements                                  |
+---------------------------------+------------+------------------------------------------+
|PO1*1*5*EA*50.00**BP*WATCH1001~  |PO1         |[PO1, 1, 5, EA, 50.00, , BP, WATCH1001~ ] |
|PO1*2*3*EA*30.00**BP*WALLET1001~ |PO1         |[PO1, 2, 3, EA, 30.00, , BP, WALLET1001~ ]|
+---------------------------------+------------+------------------------------------------+



In [0]:
elements_df.printSchema()

root
 |-- edi_segment: string (nullable = true)
 |-- segment_type: string (nullable = true)
 |-- elements: array (nullable = true)
 |    |-- element: string (containsNull = false)



In [0]:
from pyspark.sql.functions import col
po1_structured_df = po1_df.select(
    col("elements")[1].alias("line_number"),
    col("elements")[2].alias("quantity"),
    col("elements")[3].alias("uom"),
    col("elements")[4].alias("unit_price"),
    col("elements")[7].alias("product_id")
)

po1_structured_df.show(truncate=False)

+-----------+--------+---+----------+------------+
|line_number|quantity|uom|unit_price|product_id  |
+-----------+--------+---+----------+------------+
|1          |5       |EA |50.00     |WATCH1001~  |
|2          |3       |EA |30.00     |WALLET1001~ |
+-----------+--------+---+----------+------------+



In [0]:
from pyspark.sql.functions import regexp_replace

po1_structured_df = po1_structured_df.withColumn(
    "product_id",
    regexp_replace("product_id", "~", "")
)

po1_structured_df.show(truncate=False)

+-----------+--------+---+----------+-----------+
|line_number|quantity|uom|unit_price|product_id |
+-----------+--------+---+----------+-----------+
|1          |5       |EA |50.00     |WATCH1001  |
|2          |3       |EA |30.00     |WALLET1001 |
+-----------+--------+---+----------+-----------+



In [0]:
from pyspark.sql.functions import col

po1_structured_df = po1_structured_df.withColumn(
    "quantity",
    col("quantity").cast("integer")
).withColumn(
    "unit_price",
    col("unit_price").cast("decimal(10,2)")
)

po1_structured_df.printSchema()

root
 |-- line_number: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- uom: string (nullable = true)
 |-- unit_price: decimal(10,2) (nullable = true)
 |-- product_id: string (nullable = true)



In [0]:
po1_structured_df = po1_structured_df.withColumn(
    "line_total",
    col("quantity") * col("unit_price")
)

po1_structured_df.show()

+-----------+--------+---+----------+-----------+----------+
|line_number|quantity|uom|unit_price| product_id|line_total|
+-----------+--------+---+----------+-----------+----------+
|          1|       5| EA|     50.00| WATCH1001 |    250.00|
|          2|       3| EA|     30.00|WALLET1001 |     90.00|
+-----------+--------+---+----------+-----------+----------+



In [0]:
beg_df = elements_df.filter(
    elements_df["segment_type"] == "BEG"
)

beg_df.show(truncate = False)

+--------------------------------+------------+---------------------------------------+
|edi_segment                     |segment_type|elements                               |
+--------------------------------+------------+---------------------------------------+
|BEG*00*NE*PO20260806**20260806~ |BEG         |[BEG, 00, NE, PO20260806, , 20260806~ ]|
+--------------------------------+------------+---------------------------------------+



In [0]:
beg_structured_df = beg_df.select(

    col("elements")[3].alias("po_number"),
    col("elements")[5].alias("po_date")
)

beg_structured_df.show(truncate=False)


+----------+----------+
|po_number |po_date   |
+----------+----------+
|PO20260806|20260806~ |
+----------+----------+



In [0]:
beg_structured_df = beg_structured_df.withColumn(
    "po_date",
    regexp_replace("po_date", "~", "")
)
beg_structured_df.show(truncate = False)

+----------+---------+
|po_number |po_date  |
+----------+---------+
|PO20260806|20260806 |
+----------+---------+



In [0]:
from pyspark.sql.functions import trim

beg_structured_df = beg_structured_df.withColumn(
    "po_date",
    trim("po_date")
)
beg_structured_df.show(truncate = False)

+----------+--------+
|po_number |po_date |
+----------+--------+
|PO20260806|20260806|
+----------+--------+



In [0]:
from pyspark.sql.functions import to_date

beg_structured_df = beg_structured_df.withColumn(
    "po_date",
    to_date("po_date", "yyyyMMdd")
)

beg_structured_df.printSchema()

root
 |-- po_number: string (nullable = true)
 |-- po_date: date (nullable = true)



In [0]:
beg_structured_df.show()

+----------+----------+
| po_number|   po_date|
+----------+----------+
|PO20260806|2026-08-06|
+----------+----------+



In [0]:
n1_df = elements_df.filter(
    elements_df["segment_type"]=="N1"

)
n1_df.show(truncate = False)

+-------------------------------+------------+-----------------------------------+
|edi_segment                    |segment_type|elements                           |
+-------------------------------+------------+-----------------------------------+
|N1*BY*NORTHSTAR RETAIL~        |N1          |[N1, BY, NORTHSTAR RETAIL~ ]       |
|N1*ST*NORTHSTAR STORE TORONTO~ |N1          |[N1, ST, NORTHSTAR STORE TORONTO~ ]|
+-------------------------------+------------+-----------------------------------+



In [0]:
n1_structured_df = n1_df.select(
    col("elements")[1].alias("entity_type"),
    col("elements")[2].alias("entity_name")
)

n1_structured_df.show(truncate=False)

+-----------+-------------------------+
|entity_type|entity_name              |
+-----------+-------------------------+
|BY         |NORTHSTAR RETAIL~        |
|ST         |NORTHSTAR STORE TORONTO~ |
+-----------+-------------------------+



In [0]:
n1_structured_df = n1_structured_df.withColumn(
    "entity_name",
    trim(regexp_replace("entity_name", "~", ""))
)

n1_structured_df.show(truncate=False)

+-----------+-----------------------+
|entity_type|entity_name            |
+-----------+-----------------------+
|BY         |NORTHSTAR RETAIL       |
|ST         |NORTHSTAR STORE TORONTO|
+-----------+-----------------------+



In [0]:
ref_df = elements_df.filter(
    col("segment_type") == "REF"
)

ref_df.show(truncate=False)

+------------+------------+----------------+
|edi_segment |segment_type|elements        |
+------------+------------+----------------+
|REF*DP*001~ |REF         |[REF, DP, 001~ ]|
+------------+------------+----------------+



In [0]:
ref_structured_df = ref_df.select(
    col("elements")[1].alias("ref_qualifier"),
    col("elements")[2].alias("ref_value")
)

ref_structured_df.show(truncate=False)

+-------------+---------+
|ref_qualifier|ref_value|
+-------------+---------+
|DP           |001~     |
+-------------+---------+



In [0]:
ref_structured_df = ref_structured_df.withColumn(
    "ref_value",
    regexp_replace("ref_value", "~", "")
)

ref_structured_df.show(truncate=False)

+-------------+---------+
|ref_qualifier|ref_value|
+-------------+---------+
|DP           |001      |
+-------------+---------+



In [0]:
ctt_df = elements_df.filter(
    col("segment_type") == "CTT"
)

ctt_df.show(truncate=False)

+-----------+------------+----------+
|edi_segment|segment_type|elements  |
+-----------+------------+----------+
|CTT*2~     |CTT         |[CTT, 2~ ]|
+-----------+------------+----------+



In [0]:
ctt_structured_df = ctt_df.select(
    col("elements")[1].alias("transaction_line_count")
)

ctt_structured_df.show(truncate=False)

+----------------------+
|transaction_line_count|
+----------------------+
|2~                    |
+----------------------+



In [0]:
ctt_structured_df = ctt_structured_df.withColumn(
    "transaction_line_count",
    regexp_replace("transaction_line_count", "~", "").cast("integer")
)

ctt_structured_df.show()

+----------------------+
|transaction_line_count|
+----------------------+
|                     2|
+----------------------+



In [0]:
st_df = elements_df.filter(
    col("segment_type") == "ST"
)

st_df.show(truncate=False)

+-------------+------------+-----------------+
|edi_segment  |segment_type|elements         |
+-------------+------------+-----------------+
|ST*850*0001~ |ST          |[ST, 850, 0001~ ]|
+-------------+------------+-----------------+



In [0]:
st_structured_df = st_df.select(
    col("elements")[1].alias("transaction_type"),
    col("elements")[2].alias("transaction_control_number")
)

st_structured_df.show(truncate=False)

+----------------+--------------------------+
|transaction_type|transaction_control_number|
+----------------+--------------------------+
|850             |0001~                     |
+----------------+--------------------------+



In [0]:
st_structured_df = st_structured_df.withColumn(
    "transaction_control_number",
    regexp_replace("transaction_control_number", "~", "")
)

st_structured_df.show(truncate=False)

+----------------+--------------------------+
|transaction_type|transaction_control_number|
+----------------+--------------------------+
|850             |0001                      |
+----------------+--------------------------+



In [0]:
st_structured_df = st_structured_df.withColumn(
    "transaction_control_number",
    trim("transaction_control_number")
)
st_structured_df.show(truncate=False)

+----------------+--------------------------+
|transaction_type|transaction_control_number|
+----------------+--------------------------+
|850             |0001                      |
+----------------+--------------------------+



In [0]:
se_df = elements_df.filter(
    col("segment_type") == "SE"
)

se_df.show(truncate=False)

+-----------+------------+--------------+
|edi_segment|segment_type|elements      |
+-----------+------------+--------------+
|SE*9*0001~ |SE          |[SE, 9, 0001~]|
+-----------+------------+--------------+



In [0]:
se_structured_df = se_df.select(
    col("elements")[1].alias("segment_count"),
    col("elements")[2].alias("transaction_control_number")
)

se_structured_df.show(truncate=False)

+-------------+--------------------------+
|segment_count|transaction_control_number|
+-------------+--------------------------+
|9            |0001~                     |
+-------------+--------------------------+



In [0]:
se_structured_df = se_structured_df.withColumn(
    "transaction_control_number",
    regexp_replace("transaction_control_number", "~", "")
)

se_structured_df.show(truncate=False)

+-------------+--------------------------+
|segment_count|transaction_control_number|
+-------------+--------------------------+
|9            |0001                      |
+-------------+--------------------------+



In [0]:
se_structured_df = se_structured_df.withColumn(
    "transaction_control_number",
    trim("transaction_control_number")
)
se_structured_df.show(truncate=False)

+-------------+--------------------------+
|segment_count|transaction_control_number|
+-------------+--------------------------+
|9            |0001                      |
+-------------+--------------------------+



In [0]:
st_control = st_structured_df.first()["transaction_control_number"]
se_control = se_structured_df.first()["transaction_control_number"]

print("ST Control Number:", st_control)
print("SE Control Number:", se_control)

if st_control == se_control:
    print("ST/SE validation: PASSED")
else:
    print("ST/SE validation: FAILED")

ST Control Number: 0001
SE Control Number: 0001
ST/SE validation: PASSED


repr() is useful because it makes hidden spaces visible.

In [0]:
print("ST:", repr(st_control))
print("SE:", repr(se_control))

st_structured_df.show(truncate=False)
se_structured_df.show(truncate=False)

ST: '0001'
SE: '0001'
+----------------+--------------------------+
|transaction_type|transaction_control_number|
+----------------+--------------------------+
|850             |0001                      |
+----------------+--------------------------+

+-------------+--------------------------+
|segment_count|transaction_control_number|
+-------------+--------------------------+
|9            |0001                      |
+-------------+--------------------------+



In [0]:
gs_df = elements_df.filter(
    col("segment_type") == "GS"
)

gs_df.show(truncate=False)

+----------------------------------------------------+------------+--------------------------------------------------------------+
|edi_segment                                         |segment_type|elements                                                      |
+----------------------------------------------------+------------+--------------------------------------------------------------+
|GS*PO*SENDERID*RECEIVERID*20260806*1000*1*X*004010~ |GS          |[GS, PO, SENDERID, RECEIVERID, 20260806, 1000, 1, X, 004010~ ]|
+----------------------------------------------------+------------+--------------------------------------------------------------+



In [0]:
gs_structured_df = gs_df.select(
    col("elements")[6].alias("group_control_number")
)

gs_structured_df.show(truncate=False)

+--------------------+
|group_control_number|
+--------------------+
|1                   |
+--------------------+



In [0]:
gs_structured_df = gs_structured_df.withColumn(
    "group_control_number",
    regexp_replace("group_control_number", "~", ""),


)

gs_structured_df.show(truncate=False)

+--------------------+
|group_control_number|
+--------------------+
|1                   |
+--------------------+



In [0]:
gs_structured_df = gs_structured_df.withColumn(
    "group_control_number",
    trim("group_control_number")
)
gs_structured_df.show(truncate=False)

+--------------------+
|group_control_number|
+--------------------+
|1                   |
+--------------------+



In [0]:
ge_df = elements_df.filter(
    col("segment_type") == "GE"
)

ge_df.show(truncate=False)

+-----------+------------+-----------+
|edi_segment|segment_type|elements   |
+-----------+------------+-----------+
|GE*1*1~    |GE          |[GE, 1, 1~]|
+-----------+------------+-----------+



In [0]:
ge_structured_df = ge_df.select(
    col("elements")[2].alias("group_control_number")
)

ge_structured_df.show(truncate=False)

+--------------------+
|group_control_number|
+--------------------+
|1~                  |
+--------------------+



In [0]:
ge_structured_df = ge_structured_df.withColumn(
    "group_control_number",
    trim(regexp_replace("group_control_number", "~", ""))
)

ge_structured_df.show(truncate=False)

+--------------------+
|group_control_number|
+--------------------+
|1                   |
+--------------------+



In [0]:
gs_control = gs_structured_df.first()["group_control_number"]
ge_control = ge_structured_df.first()["group_control_number"]

print("GS Control Number:", repr(gs_control))
print("GE Control Number:", repr(ge_control))

if gs_control == ge_control:
    print("GS/GE validation: PASSED")
else:
    print("GS/GE validation: FAILED")

GS Control Number: '1'
GE Control Number: '1'
GS/GE validation: PASSED


In [0]:
isa_df = elements_df.filter(
    col("segment_type") == "ISA"
)

isa_df.show(truncate=False)

+---------------------------------------------------------------------------------------------------------------+------------+---------------------------------------------------------------------------------------------------------------------------------+
|edi_segment                                                                                                    |segment_type|elements                                                                                                                         |
+---------------------------------------------------------------------------------------------------------------+------------+---------------------------------------------------------------------------------------------------------------------------------+
|ISA*00*           *00*           *ZZ*SENDERID        *ZZ*RECEIVERID      *260806*1000*U*00401*000000001*0*T*>~ |ISA         |[ISA, 00,            , 00,            , ZZ, SENDERID        , ZZ, RECEIVERID      , 260806, 1000, U, 00

In [0]:
isa_structured_df = isa_df.select(
    col("elements")[13].alias("interchange_control_number")
)

isa_structured_df.show(truncate=False)

+--------------------------+
|interchange_control_number|
+--------------------------+
|000000001                 |
+--------------------------+



In [0]:
iea_df = elements_df.filter(
    col("segment_type") == "IEA"
)

iea_df.show(truncate=False)

+----------------+------------+--------------------+
|edi_segment     |segment_type|elements            |
+----------------+------------+--------------------+
|IEA*1*000000001~|IEA         |[IEA, 1, 000000001~]|
+----------------+------------+--------------------+



In [0]:
iea_structured_df = iea_df.select(
    col("elements")[2].alias("interchange_control_number")
)

iea_structured_df.show(truncate=False)

+--------------------------+
|interchange_control_number|
+--------------------------+
|000000001~                |
+--------------------------+



In [0]:
iea_structured_df = iea_structured_df.withColumn(
    "interchange_control_number",
    trim(regexp_replace("interchange_control_number", "~", ""))
)

iea_structured_df.show(truncate=False)

+--------------------------+
|interchange_control_number|
+--------------------------+
|000000001                 |
+--------------------------+



In [0]:
isa_control = isa_structured_df.first()["interchange_control_number"]
iea_control = iea_structured_df.first()["interchange_control_number"]

print("ISA Control Number:", repr(isa_control))
print("IEA Control Number:", repr(iea_control))

if isa_control == iea_control:
    print("ISA/IEA validation: PASSED")
else:
    print("ISA/IEA validation: FAILED")

ISA Control Number: '000000001'
IEA Control Number: '000000001'
ISA/IEA validation: PASSED


In [0]:
actual_po1_count = po1_df.count()

ctt_count = ctt_structured_df.first()["transaction_line_count"]

print("CTT Line Count:", ctt_count)
print("Actual PO1 Count:", actual_po1_count)

if ctt_count == actual_po1_count:
    print("CTT/PO1 validation: PASSED")
else:
    print("CTT/PO1 validation: FAILED")

CTT Line Count: 2
Actual PO1 Count: 2
CTT/PO1 validation: PASSED


In [0]:
print("BEG:")
beg_structured_df.printSchema()

print("N1:")
n1_structured_df.printSchema()

print("REF:")
ref_structured_df.printSchema()

print("PO1:")
po1_structured_df.printSchema()

BEG:
root
 |-- po_number: string (nullable = true)
 |-- po_date: date (nullable = true)

N1:
root
 |-- entity_type: string (nullable = true)
 |-- entity_name: string (nullable = true)

REF:
root
 |-- ref_qualifier: string (nullable = true)
 |-- ref_value: string (nullable = true)

PO1:
root
 |-- line_number: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- uom: string (nullable = true)
 |-- unit_price: decimal(10,2) (nullable = true)
 |-- product_id: string (nullable = true)
 |-- line_total: decimal(21,2) (nullable = true)



In [0]:
print("PO1:")
po1_structured_df.printSchema()

PO1:
root
 |-- line_number: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- uom: string (nullable = true)
 |-- unit_price: decimal(10,2) (nullable = true)
 |-- product_id: string (nullable = true)
 |-- line_total: decimal(21,2) (nullable = true)



In [0]:
n1_structured_df.show(truncate=False)

+-----------+-----------------------+
|entity_type|entity_name            |
+-----------+-----------------------+
|BY         |NORTHSTAR RETAIL       |
|ST         |NORTHSTAR STORE TORONTO|
+-----------+-----------------------+



In [0]:
buyer_df = n1_structured_df.filter(
    col("entity_type") == "BY"
).select(
    col("entity_name").alias("buyer_name")
)

buyer_df.show(truncate=False)

+----------------+
|buyer_name      |
+----------------+
|NORTHSTAR RETAIL|
+----------------+



In [0]:
ship_to_df = n1_structured_df.filter(
    col("entity_type") == "ST"
).select(
    col("entity_name").alias("ship_to_name")
)

ship_to_df.show(truncate=False)

+-----------------------+
|ship_to_name           |
+-----------------------+
|NORTHSTAR STORE TORONTO|
+-----------------------+



In [0]:
n1_combined_df = buyer_df.crossJoin(ship_to_df)

n1_combined_df.show(truncate=False)

+----------------+-----------------------+
|buyer_name      |ship_to_name           |
+----------------+-----------------------+
|NORTHSTAR RETAIL|NORTHSTAR STORE TORONTO|
+----------------+-----------------------+



In [0]:
po_header_df = beg_structured_df.crossJoin(
    n1_combined_df
)

po_header_df.show(truncate=False)

+----------+----------+----------------+-----------------------+
|po_number |po_date   |buyer_name      |ship_to_name           |
+----------+----------+----------------+-----------------------+
|PO20260806|2026-08-06|NORTHSTAR RETAIL|NORTHSTAR STORE TORONTO|
+----------+----------+----------------+-----------------------+



In [0]:
final_po_df = po_header_df.crossJoin(
    po1_structured_df
)

final_po_df.show(truncate=False)

+----------+----------+----------------+-----------------------+-----------+--------+---+----------+-----------+----------+
|po_number |po_date   |buyer_name      |ship_to_name           |line_number|quantity|uom|unit_price|product_id |line_total|
+----------+----------+----------------+-----------------------+-----------+--------+---+----------+-----------+----------+
|PO20260806|2026-08-06|NORTHSTAR RETAIL|NORTHSTAR STORE TORONTO|1          |5       |EA |50.00     |WATCH1001  |250.00    |
|PO20260806|2026-08-06|NORTHSTAR RETAIL|NORTHSTAR STORE TORONTO|2          |3       |EA |30.00     |WALLET1001 |90.00     |
+----------+----------+----------------+-----------------------+-----------+--------+---+----------+-----------+----------+



In [0]:
department_df = ref_structured_df.filter(
    col("ref_qualifier") == "DP"
).select(
    col("ref_value").alias("department")
)

department_df.show(truncate=False)

+----------+
|department|
+----------+
|001       |
+----------+



In [0]:
final_po_df = final_po_df.crossJoin(
    department_df
)

final_po_df.show(truncate=False)

+----------+----------+----------------+-----------------------+-----------+--------+---+----------+-----------+----------+----------+
|po_number |po_date   |buyer_name      |ship_to_name           |line_number|quantity|uom|unit_price|product_id |line_total|department|
+----------+----------+----------------+-----------------------+-----------+--------+---+----------+-----------+----------+----------+
|PO20260806|2026-08-06|NORTHSTAR RETAIL|NORTHSTAR STORE TORONTO|1          |5       |EA |50.00     |WATCH1001  |250.00    |001       |
|PO20260806|2026-08-06|NORTHSTAR RETAIL|NORTHSTAR STORE TORONTO|2          |3       |EA |30.00     |WALLET1001 |90.00     |001       |
+----------+----------+----------------+-----------------------+-----------+--------+---+----------+-----------+----------+----------+



In [0]:
final_po_df = final_po_df.select(
    "po_number",
    "po_date",
    "buyer_name",
    "ship_to_name",
    "department",
    "line_number",
    "product_id",
    "quantity",
    "uom",
    "unit_price",
    "line_total"
)

final_po_df.show(truncate=False)

+----------+----------+----------------+-----------------------+----------+-----------+-----------+--------+---+----------+----------+
|po_number |po_date   |buyer_name      |ship_to_name           |department|line_number|product_id |quantity|uom|unit_price|line_total|
+----------+----------+----------------+-----------------------+----------+-----------+-----------+--------+---+----------+----------+
|PO20260806|2026-08-06|NORTHSTAR RETAIL|NORTHSTAR STORE TORONTO|001       |1          |WATCH1001  |5       |EA |50.00     |250.00    |
|PO20260806|2026-08-06|NORTHSTAR RETAIL|NORTHSTAR STORE TORONTO|001       |2          |WALLET1001 |3       |EA |30.00     |90.00     |
+----------+----------+----------------+-----------------------+----------+-----------+-----------+--------+---+----------+----------+



In [0]:
from pyspark.sql.functions import sum

po_total_df = final_po_df.agg(
    sum("line_total").alias("po_total")
)

po_total_df.show()

+--------+
|po_total|
+--------+
|  340.00|
+--------+



In [0]:
final_po_df.show(truncate=False)

+----------+----------+----------------+-----------------------+----------+-----------+-----------+--------+---+----------+----------+
|po_number |po_date   |buyer_name      |ship_to_name           |department|line_number|product_id |quantity|uom|unit_price|line_total|
+----------+----------+----------------+-----------------------+----------+-----------+-----------+--------+---+----------+----------+
|PO20260806|2026-08-06|NORTHSTAR RETAIL|NORTHSTAR STORE TORONTO|001       |1          |WATCH1001  |5       |EA |50.00     |250.00    |
|PO20260806|2026-08-06|NORTHSTAR RETAIL|NORTHSTAR STORE TORONTO|001       |2          |WALLET1001 |3       |EA |30.00     |90.00     |
+----------+----------+----------------+-----------------------+----------+-----------+-----------+--------+---+----------+----------+



In [0]:
final_po_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.edi_purchase_orders")

In [0]:
spark.sql("""
    SHOW TABLES IN workspace.default
""").show(truncate=False)

+--------+-------------------+-----------+
|database|tableName          |isTemporary|
+--------+-------------------+-----------+
|default |edi_purchase_orders|false      |
|default |movies             |false      |
+--------+-------------------+-----------+



In [0]:
spark.sql("""
    SELECT *
    FROM workspace.default.edi_purchase_orders
""").show(truncate=False)

+----------+----------+----------------+-----------------------+----------+-----------+-----------+--------+---+----------+----------+
|po_number |po_date   |buyer_name      |ship_to_name           |department|line_number|product_id |quantity|uom|unit_price|line_total|
+----------+----------+----------------+-----------------------+----------+-----------+-----------+--------+---+----------+----------+
|PO20260806|2026-08-06|NORTHSTAR RETAIL|NORTHSTAR STORE TORONTO|001       |1          |WATCH1001  |5       |EA |50.00     |250.00    |
|PO20260806|2026-08-06|NORTHSTAR RETAIL|NORTHSTAR STORE TORONTO|001       |2          |WALLET1001 |3       |EA |30.00     |90.00     |
+----------+----------+----------------+-----------------------+----------+-----------+-----------+--------+---+----------+----------+

